# Regression
---
## Marketing tool
L’obiettivo di questo notebook è sviluppare un modello che, analizzando i rating sia in grado di predirre il prezzo del telefono.
Questo sistema può essere utilizzato come strumento di marketing per personalizzare le offerte, migliorare l’esperienza utente e supportare decisioni strategiche basate sui dati.

### Import dataset


In [1]:
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
import matplotlib.pyplot as plt
from tensorflow import keras
import tensorflow as tf
import seaborn as sns
import pandas as pd
import numpy as np

df = pd.read_csv("./data/Mobile Reviews Sentiment.csv")

2025-12-21 18:21:26.839175: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-21 18:21:26.907956: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-21 18:21:28.470024: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/dani/Desktop/sup/machineLearn/ML_MobileReviewsAnalysis/.venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


### Pulizia dataset

In [7]:
features = ["battery_life_rating", "camera_rating", "performance_rating", "design_rating", "display_rating"]
colonne_da_tenere = features + ["price_usd"]
df_clean = df[colonne_da_tenere].copy()

df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   battery_life_rating  50000 non-null  int64  
 1   camera_rating        50000 non-null  int64  
 2   performance_rating   50000 non-null  int64  
 3   design_rating        50000 non-null  int64  
 4   display_rating       50000 non-null  int64  
 5   price_usd            50000 non-null  float64
dtypes: float64(1), int64(5)
memory usage: 2.3 MB


### Scaling

In [8]:
X = df_clean[features]
y = df_clean['price_usd']

X = StandardScaler().fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

(40000, 5) (10000, 5)
(40000,) (10000,)


### Rete neurale

In [9]:
model = keras.models.Sequential([
    keras.layers.Dense(30, activation="relu", input_shape=X_train.shape[1:]),
    keras.layers.Dense(15, activation="relu"),
    keras.layers.Dense(1)
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()

/home/dani/Desktop/sup/machineLearn/ML_MobileReviewsAnalysis/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 30)             │           180 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 15)             │           465 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            16 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 661 (2.58 KB)

 Trainable params: 661 (2.58 KB)

 Non-trainable params: 0 (0.00 B)

### Addrestramento

In [10]:
history = model.fit(X_train, y_train, epochs=20, validation_data=(X_test, y_test))

Epoch 1/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 297454.5000 - mae: 443.8326 - val_loss: 135559.9531 - val_mae: 291.4992
Epoch 2/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 110515.0234 - mae: 267.7687 - val_loss: 98891.6953 - val_mae: 258.0358
Epoch 3/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 97521.5625 - mae: 255.8900 - val_loss: 97202.8906 - val_mae: 258.4522
Epoch 4/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 96627.2891 - mae: 255.1883 - val_loss: 96611.8125 - val_mae: 256.4366
Epoch 5/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 96417.8438 - mae: 254.9347 - val_loss: 96429.1406 - val_mae: 255.5081
Epoch 6/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 96383.0781 - mae: 254.8655 - val_loss: 96481.4609 - val_mae: 256.2807
Epoch 7/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 96343.2422 - mae: 254.8130 - val_loss: 96483.2422 - val_mae: 256.2444
Epoch 8/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 96363.3047 